# Chapter 10. Actor-Critic — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter10_3_actor_critic.ipynb)

책 본문: [10.3 Actor-Critic](https://smhanlab.com/book-ml/kor/ml2/chapter10/3.html)

이 노트북은 10.3절의 핵심 — **베이스라인(어드밴티지)이 REINFORCE의
분산을 줄인다** — 를 숫자로 확인합니다:

1. **2행동 밴딧**에서 REINFORCE와 베이스라인 업데이트의 그래디언트를
   샘플 단위로 계산해, *기댓값은 같고* 샘플은 더 작음을 확인.
2. **잘못된 베이스라인**(`b=1.9`)을 써도 기댓값이 불변임을 확인.
3. **CartPole**에서 10.2절의 REINFORCE와 Actor-Critic(A2C)을
   300 에피소드 × 3 시드씩 학습시켜 리턴 곡선을 비교.

numpy/matplotlib/torch/cpu만 씁니다 — 외부 데이터 다운로드 불필요.

## 1. 환경 준비

In [1]:
import math, random
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import gymnasium as gym

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("torch", torch.__version__, "| gymnasium", gym.__version__)
print(f"그림 저장 위치: {IMG}")

torch 2.13.0+cpu | gymnasium 1.3.0
그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 2. 장난감 2행동 밴딧: 베이스라인이 분산만 줄인다

본문과 똑같은 예: action 0은 보상 `+1.0`, action 1은 `+1.8`
(단일 스텝 에피소드, `state_feature = 1.0`). 초기
`theta = [0, 0]`이므로 `p(a=0) = 0.5`.

참값 `V(s) = 0.5*1.0 + 0.5*1.8 = 1.4`. 소프트맥스 정책의 그래디언트는
10.2절 연습문제 1의 공식: `grad_log_pi(a=0) = [1-p0, -p1]*s`,
`grad_log_pi(a=1) = [-p0, 1-p1]*s`.

In [2]:
def softmax_policy(theta, state_feature):
    logits = [theta[0] * state_feature, theta[1] * state_feature]
    m = max(logits)
    exps = [math.exp(l - m) for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

def grad_log_pi(probs, a, s):
    if a == 0:
        return [(1 - probs[0]) * s, -probs[1] * s]
    return [-probs[0] * s, (1 - probs[1]) * s]

theta, s = [0.0, 0.0], 1.0
p0 = softmax_policy(theta, s)[0]
G = {0: 1.0, 1: 1.8}
V = p0 * G[0] + (1 - p0) * G[1]
print(f"p(a=0) = {p0:.3f},  V(s) = {V}")

# 샘플 단위 그래디언트: (a=0이 나왔을 때, a=1이 나왔을 때)
for baseline in [None, V, 1.9]:  # None = REINFORCE, V = 올바른 베이스라인, 1.9 = 틀린 베이스라인
    name = "REINFORCE" if baseline is None else f"baseline={baseline}"
    g = {}
    for a in (0, 1):
        w = G[a] if baseline is None else G[a] - baseline
        gl = grad_log_pi(softmax_policy(theta, s), a, s)
        g[a] = [w * x for x in gl]
    # 기댓값 = p0 * (a=0 샘플) + p1 * (a=1 샘플)
    expv = [p0 * g[0][i] + (1 - p0) * g[1][i] for i in range(2)]
    print(f"{name:14s} a=0 -> [{g[0][0]:+.3f}, {g[0][1]:+.3f}]   "
          f"a=1 -> [{g[1][0]:+.3f}, {g[1][1]:+.3f}]   기대값 = [{expv[0]:+.3f}, {expv[1]:+.3f}]")

print()
print("세 줄의 기대값이 전부 [-0.2, +0.2]로 같음 -> 기댓값(진짜 기울기)은 베이스라인과 무관")
print("반면 REINFORCE 샘플은 [-0.9~+0.9] 폭으로 흔들리고, V를 빼면 [-0.2, +0.2]로 압축됨")

p(a=0) = 0.500,  V(s) = 1.4
REINFORCE      a=0 -> [+0.500, -0.500]   a=1 -> [-0.900, +0.900]   기대값 = [-0.200, +0.200]
baseline=1.4   a=0 -> [-0.200, +0.200]   a=1 -> [-0.200, +0.200]   기대값 = [-0.200, +0.200]
baseline=1.9   a=0 -> [-0.450, +0.450]   a=1 -> [+0.050, -0.050]   기대값 = [-0.200, +0.200]

세 줄의 기대값이 전부 [-0.2, +0.2]로 같음 -> 기댓값(진짜 기울기)은 베이스라인과 무관
반면 REINFORCE 샘플은 [-0.9~+0.9] 폭으로 흔들리고, V를 빼면 [-0.2, +0.2]로 압축됨


## 3. CartPole: REINFORCE vs A2C (300 에피소드 × 3 시드)

10.2절의 `PolicyNet`/REINFORCE(경우의 return 정규화 포함,
`lr=1e-2`)와, 10.3절의 `ActorCriticNet`/A2C(`lr=1e-3`,
Actor: `-sum log pi * A_t`, Critic: TD(0) MSE)를 각각 학습시킨다.

In [3]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, n_actions))
    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)

class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU())
        self.actor_head = nn.Linear(64, n_actions)  # Actor: softmax 정책
        self.critic_head = nn.Linear(64, 1)         # Critic: V(s)
    def forward(self, x):
        h = self.shared(x)
        return torch.softmax(self.actor_head(h), dim=-1), self.critic_head(h).squeeze(-1)

GAMMA, N_EP = 0.99, 300

def run_episode_common(net_forward, env, seed, use_critic):
    # 한 에피소드 실행. actor forward(와 critic 값)를 수집
    s, _ = env.reset(seed=seed)
    states, log_probs, rewards, nexts = [], [], [], []
    for _ in range(500):
        st = torch.tensor(s, dtype=torch.float32)
        states.append(st)
        probs = net_forward(st)
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))
        s, r, term, trunc, _ = env.step(a.item())
        rewards.append(r)
        nexts.append(torch.tensor(s, dtype=torch.float32))
        if term or trunc:
            break
    return states, log_probs, rewards, nexts

def returns_of(rewards, gamma):
    G, rets = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        rets.insert(0, G)
    return rets

def run_reinforce(seed, lr=1e-2):
    torch.manual_seed(seed)
    policy = PolicyNet(4, 2)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    env = gym.make("CartPole-v1")
    rets = []
    for ep in range(N_EP):
        states, log_probs, rewards, _ = run_episode_common(
            lambda st: policy(st), env, seed * 1000 + ep, use_critic=False)
        rets_ep = torch.tensor(returns_of(rewards, GAMMA))
        rets_ep = (rets_ep - rets_ep.mean()) / (rets_ep.std() + 1e-8)  # 10.2절의 정규화
        loss = -sum(lp * g for lp, g in zip(log_probs, rets_ep))
        opt.zero_grad(); loss.backward(); opt.step()
        rets.append(sum(rewards))
    env.close()
    return rets

def run_a2c(seed, lr=1e-3):
    torch.manual_seed(seed)
    net = ActorCriticNet(4, 2)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    env = gym.make("CartPole-v1")
    rets = []
    for ep in range(N_EP):
        states, log_probs, rewards, nexts = run_episode_common(
            lambda st: net(st)[0], env, seed * 1000 + ep, use_critic=True)
        rets_ep = torch.tensor(returns_of(rewards, GAMMA))
        vs = torch.stack([net(st)[1] for st in states])
        adv = rets_ep - vs.detach()            # A_t = G_t - V(s_t), 상수로 취급
        actor_loss = -sum(lp * ad for lp, ad in zip(log_probs, adv))
        opt.zero_grad(); actor_loss.backward(); opt.step()
        with torch.no_grad():
            vnext = torch.stack([net(ns)[1] for ns in nexts])
        critic_loss = nn.functional.mse_loss(vs, torch.tensor(rewards) + GAMMA * vnext)
        opt.zero_grad(); critic_loss.backward(); opt.step()
        rets.append(sum(rewards))
    env.close()
    return rets

In [4]:
SEEDS = [0, 1, 2]
re_runs = {s: run_reinforce(s) for s in SEEDS}
a2c_runs = {s: run_a2c(s) for s in SEEDS}
print("학습 완료")
for s in SEEDS:
    print(f"seed {s}: REINFORCE last20={sum(re_runs[s][-20:]) / 20:.1f}, "
          f"A2C last20={sum(a2c_runs[s][-20:]) / 20:.1f}")

학습 완료
seed 0: REINFORCE last20=99.5, A2C last20=211.3
seed 1: REINFORCE last20=251.4, A2C last20=177.9
seed 2: REINFORCE last20=151.9, A2C last20=184.2


## 4. 학습 곡선: 시드 간 일관성(분산)의 차이

20에피소드 이동평균을 그린다. REINFORCE 곡선은 시드마다 크게 갈리고,
A2C 곡선은 세 시드가 비슷한 높이로 수렴함을 확인한다. 오른쪽은 마지막
20 에피소드 평균 리턴의 시드 간 박스플롯 — A2C의 박스가 더 좁다.

In [5]:
def moving_average(xs, w=20):
    return [sum(xs[max(0, i - w + 1):i + 1]) / min(i + 1, w) for i in range(len(xs))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for s in SEEDS:
    axes[0].plot(moving_average(re_runs[s]), alpha=0.55, label=f"REINFORCE (seed {s})")
    axes[0].plot(moving_average(a2c_runs[s]), alpha=0.9, label=f"A2C (seed {s})")
axes[0].axhline(250, color="gray", linestyle=":", linewidth=1)
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return (20-episode moving average)")
axes[0].set_title("CartPole: REINFORCE vs A2C (300 episodes, 3 seeds)")
axes[0].legend(fontsize=8)

last20_re = [sum(re_runs[s][-20:]) / 20 for s in SEEDS]
last20_a2c = [sum(a2c_runs[s][-20:]) / 20 for s in SEEDS]
axes[1].boxplot([last20_re, last20_a2c], tick_labels=["REINFORCE", "A2C"])
axes[1].set_ylabel("Mean return over last 20 episodes")
axes[1].set_title("Variation across seeds (box = IQR, whiskers = min~max)")

plt.tight_layout()
plt.savefig(os.path.join(IMG, "ch10_3_reinforce_vs_a2c.svg"), bbox_inches="tight")
plt.show()
print(f"SVG 저장: {os.path.join(IMG, 'ch10_3_reinforce_vs_a2c.svg')}")
print(f"REINFORCE last20 = {[round(x, 1) for x in last20_re]}  (시드 간 폭 {max(last20_re) - min(last20_re):.1f})")
print(f"A2C       last20 = {[round(x, 1) for x in last20_a2c]}  (시드 간 폭 {max(last20_a2c) - min(last20_a2c):.1f})")

SVG 저장: /home/smhan/book-ml/kor/src/images/ch10_3_reinforce_vs_a2c.svg
REINFORCE last20 = [99.5, 251.4, 151.9]  (시드 간 폭 152.0)
A2C       last20 = [211.3, 177.9, 184.2]  (시드 간 폭 33.4)


## 5. 정리

| 확인한 것 | 결론 |
|---|---|
| 밴딧: `b` = None / `V` / `1.9` | 그래디언트 **기댓값**은 셋 다 `[-0.2, +0.2]` (불변) |
| 밴딧: 샘플 단위 | `V`를 빼면 샘플이 `[-0.2, +0.2]`로 압축 (REINFORCE는 `[-0.9~+0.9]`) |
| CartPole last20 시드 간 폭 | REINFORCE가 A2C보다 넓음 — **분산 감소 = 시드 간 일관성** |
| 도달 속도 | 이 실험(300에피소드, 3시드)에서는 REINFORCE가 더 빠른 시드도 있었음 — 분산 감소의 *안정성* 이득은 PPO(Chapter 11)에서 온전히 수확됨 |

> "절대 보상(`G_t`)"이 아니라 "평균 대비 향상(`A_t = G_t - V(s_t)`)으로
> 배우는" 이 한 줄이 10.3절의 전부 — 그리고 Chapter 11의 PPO가 바로
> 이 `A_t`를 클리핑된 목적함수에 재사용한다.